In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-04-14 17:21:59.069452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776187319.302862      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776187319.363262      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776187319.897188      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776187319.897265      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776187319.897281      22 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train.npy'))
id_train = np.load(os.path.join(INPUT_PATH, 'id_train.npy'), allow_pickle=True)

X_val = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val.npy'))
id_val = np.load(os.path.join(INPUT_PATH, 'id_val.npy'), allow_pickle=True)

X_test = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test.npy'))
id_test = np.load(os.path.join(INPUT_PATH, 'id_test.npy'), allow_pickle=True)

In [4]:
print("Train:", X_train.shape, y_train.shape, id_train.shape)
print("Val  :", X_val.shape, y_val.shape, id_val.shape)
print("Test :", X_test.shape, y_test.shape, id_test.shape)

print("Unique train patients:", len(np.unique(id_train)))
print("Unique val patients  :", len(np.unique(id_val)))
print("Unique test patients :", len(np.unique(id_test)))

Train: (145772, 10, 133) (145772,) (145772,)
Val  : (36597, 10, 133) (36597,) (36597,)
Test : (45552, 10, 133) (45552,) (45552,)
Unique train patients: 25456
Unique val patients  : 6352
Unique test patients : 7964


In [5]:
X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


### FIXED HYPERPARAMETERS FOR ENSEMBLE

In [6]:
BEST_UNITS = 16
BEST_DROPOUT = 0.3
BEST_BATCH_SIZE = 512

N_MODELS = 5
SEEDS = [42, 52, 62, 72, 82]
EPOCHS = 30

print("Ensemble configuration:")
print(f"BEST_UNITS      = {BEST_UNITS}")
print(f"BEST_DROPOUT    = {BEST_DROPOUT}")
print(f"BEST_BATCH_SIZE = {BEST_BATCH_SIZE}")
print(f"N_MODELS        = {N_MODELS}")
print(f"SEEDS           = {SEEDS}")
print(f"EPOCHS          = {EPOCHS}")

Ensemble configuration:
BEST_UNITS      = 16
BEST_DROPOUT    = 0.3
BEST_BATCH_SIZE = 512
N_MODELS        = 5
SEEDS           = [42, 52, 62, 72, 82]
EPOCHS          = 30


In [7]:
from collections import defaultdict
# ===== config =====
NEG_SUBSET_FRAC = 0.9   # chỉ subset trên negative patients
PATIENT_SUBSET_SEED = 42

assert len(SEEDS) == N_MODELS, "SEEDS và N_MODELS phải khớp nhau"

# ===== patient-level label =====
# nếu 1 patient có ít nhất 1 sequence label = 1 thì xem là positive patient
patient_labels = defaultdict(int)

for pid, y in zip(id_train, y_train):
    patient_labels[pid] = max(patient_labels[pid], int(y))

train_patient_ids = np.array(sorted(patient_labels.keys()))
pos_patients = np.array([pid for pid in train_patient_ids if patient_labels[pid] == 1])
neg_patients = np.array([pid for pid in train_patient_ids if patient_labels[pid] == 0])

print("Total train patients:", len(train_patient_ids))
print("Positive patients   :", len(pos_patients))
print("Negative patients   :", len(neg_patients))

def make_patient_subsets_all_pos(
    pos_patients,
    neg_patients,
    n_models=5,
    neg_frac=0.9,
    seed=42
):
    subsets = []
    neg_counts = []

    n_neg = max(1, int(len(neg_patients) * neg_frac))

    for i in range(n_models):
        rng = np.random.default_rng(seed + i)

        # Giữ toàn bộ positive patients cho mọi member
        sub_pos = np.array(pos_patients, copy=True)

        # Chỉ subset negative patients để tạo diversity
        sub_neg = rng.choice(neg_patients, size=n_neg, replace=False)

        subset_ids = np.concatenate([sub_pos, sub_neg])
        rng.shuffle(subset_ids)

        subsets.append(subset_ids)
        neg_counts.append(len(sub_neg))

    return subsets, n_neg, neg_counts

patient_subsets, n_neg_per_member, neg_counts = make_patient_subsets_all_pos(
    pos_patients=pos_patients,
    neg_patients=neg_patients,
    n_models=N_MODELS,
    neg_frac=NEG_SUBSET_FRAC,
    seed=PATIENT_SUBSET_SEED
)

print("\nImproved patient-subset strategy:")
print("- Keep ALL positive patients for every member")
print(f"- Subset only negative patients with NEG_SUBSET_FRAC = {NEG_SUBSET_FRAC}")
print(f"- Negative patients per member = {n_neg_per_member}")

for i, subset_ids in enumerate(patient_subsets):
    unique_ids = np.unique(subset_ids)
    n_pos_in_subset = np.intersect1d(unique_ids, pos_patients).size
    n_neg_in_subset = np.intersect1d(unique_ids, neg_patients).size
    print(
        f"Model {i+1}: total={len(unique_ids)} patients | "
        f"pos={n_pos_in_subset} | neg={n_neg_in_subset}"
    )

Total train patients: 25456
Positive patients   : 1650
Negative patients   : 23806

Improved patient-subset strategy:
- Keep ALL positive patients for every member
- Subset only negative patients with NEG_SUBSET_FRAC = 0.9
- Negative patients per member = 21425
Model 1: total=23075 patients | pos=1650 | neg=21425
Model 2: total=23075 patients | pos=1650 | neg=21425
Model 3: total=23075 patients | pos=1650 | neg=21425
Model 4: total=23075 patients | pos=1650 | neg=21425
Model 5: total=23075 patients | pos=1650 | neg=21425


In [8]:
def create_model(n_lstm_units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(n_lstm_units, return_sequences=True),
            input_shape=(X_train.shape[1], X_train.shape[2])
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Bidirectional(
            LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_model")

✅ Đã khởi tạo create_model


In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}

print("Trọng số lớp Train:", class_weights_dict)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}


### TRAIN ENSEMBLE MEMBERS

In [10]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional

val_probs_list = []
test_probs_list = []
member_summaries = []

for member_idx, seed in enumerate(SEEDS, start=1):
    print("\n" + "="*70)
    print(f"TRAINING ENSEMBLE MEMBER {member_idx}/{N_MODELS} | seed={seed}")
    print("="*70)

    tf.keras.utils.set_random_seed(seed)

    # 1) LẤY IMPROVED PATIENT SUBSET CHO MEMBER HIỆN TẠI
    #    - giữ toàn bộ positive patients
    #    - chỉ subset negative patients
    subset_patient_ids = patient_subsets[member_idx - 1]

    # Mask các sequence thuộc các patient trong subset này
    train_mask = np.isin(id_train, subset_patient_ids)

    X_train_sub = X_train[train_mask]
    y_train_sub = y_train[train_mask]
    id_train_sub = id_train[train_mask]

    unique_subset_ids = np.unique(id_train_sub)
    n_pos_subset = np.intersect1d(unique_subset_ids, pos_patients).size
    n_neg_subset = np.intersect1d(unique_subset_ids, neg_patients).size

    print(f"Subset patients : {len(unique_subset_ids)}")
    print(f"  - positive patients kept : {n_pos_subset}/{len(pos_patients)}")
    print(f"  - negative patients used : {n_neg_subset}/{len(neg_patients)}")
    print(f"Subset samples  : {len(X_train_sub)}")
    print(f"Positive rate   : {y_train_sub.mean():.6f}")

    # 2) TÍNH CLASS WEIGHT RIÊNG CHO SUBSET HIỆN TẠI
    subset_classes = np.unique(y_train_sub)
    subset_class_weights = compute_class_weight(
        class_weight='balanced',
        classes=subset_classes,
        y=y_train_sub
    )
    class_weights_dict_sub = {
        int(cls): float(w) for cls, w in zip(subset_classes, subset_class_weights)
    }

    print("Class weights   :", class_weights_dict_sub)

    # 3) TẠO MODEL
    model = create_model(
        n_lstm_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT
    )

    # 4) CALLBACK CHO TỪNG MEMBER
    early_stopping = EarlyStopping(
        monitor='val_auprc',
        mode='max',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_auprc',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )

    # 5) TRAIN TRÊN IMPROVED SUBSET
    history = model.fit(
        X_train_sub, y_train_sub,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BEST_BATCH_SIZE,
        class_weight=class_weights_dict_sub,
        shuffle=True,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    # 6) PREDICT TRÊN VAL / TEST GIỮ NGUYÊN
    val_prob = model.predict(X_val, verbose=0).ravel()
    test_prob = model.predict(X_test, verbose=0).ravel()

    val_probs_list.append(val_prob)
    test_probs_list.append(test_prob)

    # 7) LẤY BEST EPOCH THEO val_auprc
    if "val_auprc" in history.history and len(history.history["val_auprc"]) > 0:
        best_epoch = int(np.argmax(history.history["val_auprc"]) + 1)
        best_idx = best_epoch - 1
        best_val_auprc = float(np.max(history.history["val_auprc"]))
    else:
        best_epoch = None
        best_idx = -1
        best_val_auprc = np.nan

    # 8) LƯU THÔNG TIN TỪNG MEMBER
    member_summary = {
        "member": member_idx,
        "seed": seed,
        "n_subset_patients": len(unique_subset_ids),
        "n_positive_patients": int(n_pos_subset),
        "n_negative_patients": int(n_neg_subset),
        "n_subset_samples": len(X_train_sub),
        "positive_rate": float(y_train_sub.mean()),
        "best_epoch": best_epoch,
        "val_loss": history.history["val_loss"][best_idx] if "val_loss" in history.history and best_idx >= 0 else np.nan,
        "val_accuracy": history.history["val_accuracy"][best_idx] if "val_accuracy" in history.history and best_idx >= 0 else np.nan,
        "val_auroc": history.history["val_auroc"][best_idx] if "val_auroc" in history.history and best_idx >= 0 else np.nan,
        "val_auprc": history.history["val_auprc"][best_idx] if "val_auprc" in history.history and best_idx >= 0 else np.nan,
        "val_recall": history.history["val_recall"][best_idx] if "val_recall" in history.history and best_idx >= 0 else np.nan,
        "val_precision": history.history["val_precision"][best_idx] if "val_precision" in history.history and best_idx >= 0 else np.nan,
        "best_val_auprc_in_history": best_val_auprc
    }

    member_summaries.append(member_summary)

    print("\nValidation metrics of this member (from history at best epoch):")
    for key in ["val_loss", "val_accuracy", "val_auroc", "val_auprc", "val_recall", "val_precision"]:
        if key in history.history and best_idx >= 0:
            print(f"{key}: {history.history[key][best_idx]:.4f}")

   
# 9) GHÉP XÁC SUẤT CỦA CÁC MEMBER
val_probs_array = np.vstack(val_probs_list)    # shape: (N_MODELS, len(X_val))
test_probs_array = np.vstack(test_probs_list)  # shape: (N_MODELS, len(X_test))

df_members = pd.DataFrame(member_summaries)

print("\n" + "="*70)
print("IMPROVED PATIENT-SUBSET ENSEMBLE TRAINING FINISHED")
print("="*70)
print("val_probs_array shape :", val_probs_array.shape)
print("test_probs_array shape:", test_probs_array.shape)
print("\nMember summary:")
print(df_members)


TRAINING ENSEMBLE MEMBER 1/5 | seed=42
Subset patients : 23075
  - positive patients kept : 1650/1650
  - negative patients used : 21425/23806
Subset samples  : 133522
Positive rate   : 0.082204
Class weights   : {0: 0.5447831834576404, 1: 6.082452623906706}


I0000 00:00:1776187361.713150      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776187361.719147      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1776187369.950722      72 cuda_dnn.cc:529] Loaded cuDNN version 91002


261/261 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.5159 - auprc: 0.1894 - auroc: 0.6951 - loss: 0.6431 - precision: 0.1252 - recall: 0.7674 - val_accuracy: 0.8120 - val_auprc: 0.3115 - val_auroc: 0.8361 - val_loss: 0.4469 - val_precision: 0.2454 - val_recall: 0.7050 - learning_rate: 0.0010
Epoch 2/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7737 - auprc: 0.3400 - auroc: 0.8528 - loss: 0.4815 - precision: 0.2399 - recall: 0.8088 - val_accuracy: 0.8176 - val_auprc: 0.3415 - val_auroc: 0.8422 - val_loss: 0.3938 - val_precision: 0.2515 - val_recall: 0.7035 - learning_rate: 0.0010
Epoch 3/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7955 - auprc: 0.3815 - auroc: 0.8819 - loss: 0.4316 - precision: 0.2663 - recall: 0.8480 - val_accuracy: 0.8313 - val_auprc: 0.3532 - val_auroc: 0.8375 - val_loss: 0.3653 - val_precision: 0.2651 - val_recall: 0.6831 - learning_rate: 0.0010
Epoch 4/30
261/261 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8138 - auprc: 0.4400 -

### AVERAGE PROBABILITIES OF ENSEMBLE MEMBERS

In [11]:
import numpy as np

# Ưu tiên weight theo validation AUPRC tại best epoch của từng member
member_weights = df_members["val_auprc"].to_numpy(dtype=float)

if np.any(np.isnan(member_weights)) or member_weights.sum() <= 0:
    print("Fallback to uniform weights because val_auprc is invalid.")
    member_weights = np.ones(len(df_members), dtype=float)

member_weights = member_weights / member_weights.sum()

print("Member weights (normalized by val_auprc):")
for idx, w in enumerate(member_weights, start=1):
    print(f"Member {idx}: {w:.4f}")

# Weighted average
ensemble_val_prob = np.average(val_probs_array, axis=0, weights=member_weights)
ensemble_test_prob = np.average(test_probs_array, axis=0, weights=member_weights)

# Tham khảo thêm mean ensemble cũ để tiện so nhanh nếu cần
ensemble_val_prob_mean = np.mean(val_probs_array, axis=0)
ensemble_test_prob_mean = np.mean(test_probs_array, axis=0)

print("\nensemble_val_prob shape :", ensemble_val_prob.shape)
print("ensemble_test_prob shape:", ensemble_test_prob.shape)

print("\nValidation probability range:")
print("min =", ensemble_val_prob.min(), "| max =", ensemble_val_prob.max())

print("\nTest probability range:")
print("min =", ensemble_test_prob.min(), "| max =", ensemble_test_prob.max())

Member weights (normalized by val_auprc):
Member 1: 0.2062
Member 2: 0.2050
Member 3: 0.1892
Member 4: 0.1978
Member 5: 0.2018

ensemble_val_prob shape : (36597,)
ensemble_test_prob shape: (45552,)

Validation probability range:
min = 0.0007636152591538946 | max = 0.9669842445317622

Test probability range:
min = 0.0007884658212241674 | max = 0.9672561313820323


### SCAN THRESHOLDS ON ENSEMBLE VALIDATION PROBABILITY

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def eval_threshold(y_true, y_prob, th):
    y_pred = (y_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    youden_j = sensitivity + specificity - 1

    return {
        "threshold": round(float(th), 2),
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "youden_j": youden_j,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

rows = []
for th in np.arange(0.05, 0.96, 0.01):
    rows.append(eval_threshold(y_val, ensemble_val_prob, th))

df_th_ensemble = pd.DataFrame(rows)

print("Top 10 threshold theo Youden's J:")
print(df_th_ensemble.sort_values("youden_j", ascending=False).head(10))

print("\nTop 10 threshold theo F1:")
print(df_th_ensemble.sort_values("f1", ascending=False).head(10))

Top 10 threshold theo Youden's J:
    threshold  accuracy  sensitivity  specificity  precision    recall  \
24       0.29  0.783452     0.796992     0.782333   0.232262  0.796992   
20       0.25  0.763888     0.818833     0.759348   0.219440  0.818833   
19       0.24  0.758395     0.824919     0.752899   0.216196  0.824919   
23       0.28  0.778342     0.801289     0.776447   0.228484  0.801289   
21       0.26  0.769079     0.811314     0.765590   0.222375  0.811314   
18       0.23  0.752685     0.830290     0.746273   0.212830  0.830290   
22       0.27  0.773588     0.805585     0.770944   0.225158  0.805585   
26       0.31  0.791896     0.783745     0.792569   0.237909  0.783745   
25       0.30  0.787469     0.788758     0.787362   0.234586  0.788758   
17       0.22  0.746564     0.835303     0.739232   0.209275  0.835303   

          f1  youden_j     tn    fp   fn    tp  
24  0.359699  0.579326  26446  7358  567  2226  
20  0.346122  0.578181  25669  8135  506  2287  
19  

### CHOOSE FINAL THRESHOLD ON VALIDATION


In [13]:
# Rule: sensitivity >= 0.80, then maximize specificity

target_sensitivity = 0.80

candidate = df_th_ensemble[df_th_ensemble["sensitivity"] >= target_sensitivity].copy()

if len(candidate) == 0:
    print("Không có threshold nào đạt sensitivity mục tiêu.")
else:
    best_row = candidate.sort_values(
        ["specificity", "youden_j", "f1"],
        ascending=False
    ).iloc[0]

    best_threshold = float(best_row["threshold"])

    print("Best threshold for ensemble:")
    print(best_row)

    print(f"\nChosen best_threshold = {best_threshold:.2f}")

Best threshold for ensemble:
threshold          0.280000
accuracy           0.778342
sensitivity        0.801289
specificity        0.776447
precision          0.228484
recall             0.801289
f1                 0.355577
youden_j           0.577736
tn             26247.000000
fp              7557.000000
fn               555.000000
tp              2238.000000
Name: 23, dtype: float64

Chosen best_threshold = 0.28


### FINAL TEST EVALUATION FOR ENSEMBLE

In [14]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ===== 1. Threshold-independent metrics =====
test_auroc = roc_auc_score(y_test, ensemble_test_prob)
test_auprc = average_precision_score(y_test, ensemble_test_prob)

# ===== 2. Threshold-dependent prediction =====
test_pred = (ensemble_test_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
acc = accuracy_score(y_test, test_pred)
youden_j = sensitivity + specificity - 1

print(f"=== ENSEMBLE TEST RESULT WITH THRESHOLD = {best_threshold:.2f} ===")

print("\n--- Threshold-independent metrics ---")
print(f"AUROC:      {test_auroc:.4f}")
print(f"AUPRC:      {test_auprc:.4f}")

print("\n--- Threshold-dependent metrics ---")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

=== ENSEMBLE TEST RESULT WITH THRESHOLD = 0.28 ===

--- Threshold-independent metrics ---
AUROC:      0.8865
AUPRC:      0.4184

--- Threshold-dependent metrics ---
Accuracy:    0.7861
Sensitivity: 0.8451
Specificity: 0.7816
Precision:   0.2271
Recall:      0.8451
F1-score:    0.3580
Youden's J:  0.6267

Confusion Matrix:
[[33092  9245]
 [  498  2717]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9852    0.7816    0.8717     42337
           1     0.2271    0.8451    0.3580      3215

    accuracy                         0.7861     45552
   macro avg     0.6062    0.8134    0.6149     45552
weighted avg     0.9317    0.7861    0.8354     45552



### Error analysis inputs for the main ensemble model

In [15]:
import numpy as np
import pandas as pd

# ===== Build sequence-level prediction tables =====
df_val_pred = pd.DataFrame({
    "row_id": np.arange(len(y_val)),
    "patient_id": id_val,
    "y_true": y_val.astype(int),
    "y_prob": ensemble_val_prob,
})

df_test_pred = pd.DataFrame({
    "row_id": np.arange(len(y_test)),
    "patient_id": id_test,
    "y_true": y_test.astype(int),
    "y_prob": ensemble_test_prob,
})

df_val_pred["y_pred"] = (df_val_pred["y_prob"] >= best_threshold).astype(int)
df_test_pred["y_pred"] = (df_test_pred["y_prob"] >= best_threshold).astype(int)

def error_type(y_true, y_pred):
    if y_true == 1 and y_pred == 1:
        return "TP"
    elif y_true == 1 and y_pred == 0:
        return "FN"
    elif y_true == 0 and y_pred == 1:
        return "FP"
    else:
        return "TN"

df_val_pred["error_type"] = [
    error_type(t, p) for t, p in zip(df_val_pred["y_true"], df_val_pred["y_pred"])
]
df_test_pred["error_type"] = [
    error_type(t, p) for t, p in zip(df_test_pred["y_true"], df_test_pred["y_pred"])
]

# ===== Ensemble disagreement / uncertainty =====
df_val_pred["member_prob_std"] = val_probs_array.std(axis=0)
df_val_pred["member_prob_min"] = val_probs_array.min(axis=0)
df_val_pred["member_prob_max"] = val_probs_array.max(axis=0)

df_test_pred["member_prob_std"] = test_probs_array.std(axis=0)
df_test_pred["member_prob_min"] = test_probs_array.min(axis=0)
df_test_pred["member_prob_max"] = test_probs_array.max(axis=0)

print("Validation error types:")
print(df_val_pred["error_type"].value_counts())

print("\nTest error types:")
print(df_test_pred["error_type"].value_counts())

df_val_pred.to_csv("ensemble_val_predictions.csv", index=False)
df_test_pred.to_csv("ensemble_test_predictions.csv", index=False)

print("\nSaved:")
print("- ensemble_val_predictions.csv")
print("- ensemble_test_predictions.csv")

Validation error types:
error_type
TN    26247
FP     7557
TP     2238
FN      555
Name: count, dtype: int64

Test error types:
error_type
TN    33092
FP     9245
TP     2717
FN      498
Name: count, dtype: int64

Saved:
- ensemble_val_predictions.csv
- ensemble_test_predictions.csv


In [16]:
def summarize_error_groups(df):
    return (
        df.groupby("error_type")
          .agg(
              n=("row_id", "size"),
              mean_prob=("y_prob", "mean"),
              median_prob=("y_prob", "median"),
              mean_member_std=("member_prob_std", "mean"),
              n_patients=("patient_id", "nunique"),
          )
          .sort_values("n", ascending=False)
    )

print("Validation summary:")
display(summarize_error_groups(df_val_pred))

print("\nTest summary:")
display(summarize_error_groups(df_test_pred))

Validation summary:


,n,mean_prob,median_prob,mean_member_std,n_patients
error_type,,,,,
TN,26247,0.054544,0.021214,0.054152,6032
FP,7557,0.589930,0.568583,0.182194,2406
TP,2238,0.744879,0.831959,0.126913,373
FN,555,0.114836,0.101764,0.121110,143



Test summary:


,n,mean_prob,median_prob,mean_member_std,n_patients
error_type,,,,,
TN,33092,0.054526,0.021582,0.054477,7585
FP,9245,0.590943,0.568893,0.181989,3010
TP,2717,0.758884,0.843661,0.118985,452
FN,498,0.123616,0.119807,0.119547,144


### Metadata

In [17]:
meta_train = pd.read_csv('/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val/meta_train.csv')
meta_val = pd.read_csv('/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val/meta_val.csv')
meta_test = pd.read_csv('/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val/meta_test.csv')

# ===== Merge on row_id =====
df_val_analysis = df_val_pred.merge(
    meta_val,
    on='row_id',
    how='left',
    validate='one_to_one',
    suffixes=('_pred', '_meta')
)

df_test_analysis = df_test_pred.merge(
    meta_test,
    on='row_id',
    how='left',
    validate='one_to_one',
    suffixes=('_pred', '_meta')
)

# ===== Correct merge checks =====
assert len(df_val_analysis) == len(df_val_pred)
assert len(df_test_analysis) == len(df_test_pred)

assert (df_val_analysis['patient_id_pred'].values == df_val_analysis['patient_id_meta'].values).all()
assert (df_test_analysis['patient_id_pred'].values == df_test_analysis['patient_id_meta'].values).all()

assert (df_val_analysis['y_true_pred'].values == df_val_analysis['y_true_meta'].values).all()
assert (df_test_analysis['y_true_pred'].values == df_test_analysis['y_true_meta'].values).all()

# Chỉ kiểm tra các cột metadata KHÔNG được phép thiếu
assert df_val_analysis['window_end_hour'].isna().sum() == 0
assert df_test_analysis['window_end_hour'].isna().sum() == 0

assert df_val_analysis['raw_missing_rate_window'].isna().sum() == 0
assert df_test_analysis['raw_missing_rate_window'].isna().sum() == 0

# ===== Rename columns back to clean names =====
df_val_analysis = df_val_analysis.rename(columns={
    'patient_id_pred': 'patient_id',
    'y_true_pred': 'y_true'
}).drop(columns=['patient_id_meta', 'y_true_meta'])

df_test_analysis = df_test_analysis.rename(columns={
    'patient_id_pred': 'patient_id',
    'y_true_pred': 'y_true'
}).drop(columns=['patient_id_meta', 'y_true_meta'])

print("df_val_analysis:", df_val_analysis.shape)
print("df_test_analysis:", df_test_analysis.shape)

print("\nMissing check:")
print("VAL  - window_end_hour:", df_val_analysis['window_end_hour'].isna().sum())
print("VAL  - raw_missing_rate_window:", df_val_analysis['raw_missing_rate_window'].isna().sum())
print("VAL  - last_HR_raw:", df_val_analysis['last_HR_raw'].isna().sum())

print("TEST - window_end_hour:", df_test_analysis['window_end_hour'].isna().sum())
print("TEST - raw_missing_rate_window:", df_test_analysis['raw_missing_rate_window'].isna().sum())
print("TEST - last_HR_raw:", df_test_analysis['last_HR_raw'].isna().sum())

display(df_val_analysis.head())
display(df_test_analysis.head())

print("Merge check passed.")

df_val_analysis: (36597, 32)
df_test_analysis: (45552, 32)

Missing check:
VAL  - window_end_hour: 0
VAL  - raw_missing_rate_window: 0
VAL  - last_HR_raw: 3034
TEST - window_end_hour: 0
TEST - raw_missing_rate_window: 0
TEST - last_HR_raw: 3694


,row_id,patient_id,y_true,y_prob,y_pred,error_type,member_prob_std,member_prob_min,member_prob_max,seq_index_within_patient,...,last_O2Sat_raw,last_Temp_raw,delta_HR_raw,delta_Resp_raw,delta_SBP_raw,delta_MAP_raw,n_obs_HR_window,n_obs_Resp_window,n_obs_SBP_window,n_obs_MAP_window
0,0,6,0,0.172423,0,TN,0.224031,0.012284,0.610221,0,...,95.0,NaN,-26.5,-7.0,-27.50,-18.0,9,9,9,9
1,1,6,0,0.712051,1,FP,0.220240,0.313599,0.916034,1,...,94.5,NaN,-0.5,-10.0,3.75,8.5,10,10,10,10
2,2,6,0,0.789524,1,FP,0.076810,0.665866,0.897868,2,...,95.0,NaN,-2.0,7.5,-7.00,-7.0,10,10,10,10
3,3,6,0,0.747495,1,FP,0.294869,0.160944,0.931463,3,...,95.0,NaN,-8.0,-7.0,4.00,8.0,10,10,10,10
4,4,6,0,0.523092,1,FP,0.246262,0.091657,0.786037,4,...,95.0,NaN,-10.0,-11.0,16.00,8.0,10,10,10,10


,row_id,patient_id,y_true,y_prob,y_pred,error_type,member_prob_std,member_prob_min,member_prob_max,seq_index_within_patient,...,last_O2Sat_raw,last_Temp_raw,delta_HR_raw,delta_Resp_raw,delta_SBP_raw,delta_MAP_raw,n_obs_HR_window,n_obs_Resp_window,n_obs_SBP_window,n_obs_MAP_window
0,0,1,0,0.121543,0,TN,0.068882,0.058463,0.232931,0,...,95.0,36.11,-3.0,-6.5,19.0,12.0,9,9,9,9
1,1,1,0,0.153917,0,TN,0.136178,0.008704,0.372397,1,...,94.0,NaN,0.0,-3.0,-19.0,-10.0,10,10,10,10
2,2,1,0,0.030166,0,TN,0.028607,0.002905,0.083367,2,...,97.0,36.11,-1.0,6.0,-25.0,-14.0,10,7,4,10
3,3,4,0,0.004171,0,TN,0.003180,0.001184,0.009597,0,...,98.0,NaN,-21.0,-2.5,-16.5,-17.5,9,9,9,8
4,4,4,0,0.006081,0,TN,0.004795,0.000321,0.013251,1,...,99.0,36.94,3.0,-4.0,-13.0,-7.0,10,10,10,10


Merge check passed.


In [18]:
# ===== Fixed missingness cutoff from TRAIN =====
missing_cutoff = meta_train['raw_missing_rate_window'].median()
print("missing_cutoff from train =", missing_cutoff)

missing_cutoff from train = 0.1666666666666666


In [19]:
# ===== Create clinically meaningful subgroups =====

for df in [df_val_analysis, df_test_analysis]:
    df['hr_high'] = (df['last_HR_raw'] >= 100).astype(int)
    df['resp_high'] = (df['last_Resp_raw'] >= 22).astype(int)
    df['sbp_low'] = (df['last_SBP_raw'] <= 100).astype(int)
    df['map_low'] = (df['last_MAP_raw'] < 65).astype(int)

    # Dùng cutoff cố định từ TRAIN
    df['missing_high'] = (df['raw_missing_rate_window'] >= missing_cutoff).astype(int)

    df['delta_hr_up'] = (df['delta_HR_raw'] > 0).astype(int)
    df['delta_resp_up'] = (df['delta_Resp_raw'] > 0).astype(int)
    df['delta_sbp_down'] = (df['delta_SBP_raw'] < 0).astype(int)
    df['delta_map_down'] = (df['delta_MAP_raw'] < 0).astype(int)

    df['eda_pattern_strong'] = (
        (df['delta_hr_up'] == 1) &
        (df['delta_resp_up'] == 1) &
        (
            (df['delta_sbp_down'] == 1) |
            (df['delta_map_down'] == 1)
        )
    ).astype(int)

print("Done creating subgroup flags.")
print("Using fixed missing_cutoff from train:", missing_cutoff)

Done creating subgroup flags.
Using fixed missing_cutoff from train: 0.1666666666666666


In [20]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, average_precision_score

def subgroup_report(df, group_col):
    rows = []

    for group_value, sub in df.groupby(group_col):
        y_true = sub['y_true'].values
        y_prob = sub['y_prob'].values
        y_pred = sub['y_pred'].values

        if len(np.unique(y_true)) < 2:
            auprc = np.nan
        else:
            auprc = average_precision_score(y_true, y_prob)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = precision_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        rows.append({
            'group': group_value,
            'n': len(sub),
            'positive_rate': sub['y_true'].mean(),
            'AUPRC': auprc,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'precision': precision,
            'f1': f1,
            'FP': fp,
            'FN': fn,
            'TP': tp,
            'TN': tn,
            'mean_prob': sub['y_prob'].mean(),
            'mean_member_std': sub['member_prob_std'].mean()
        })

    return pd.DataFrame(rows).sort_values('group')

In [21]:
# ===== First subgroup analyses =====

report_pattern_val = subgroup_report(df_val_analysis, 'eda_pattern_strong')
report_pattern_test = subgroup_report(df_test_analysis, 'eda_pattern_strong')

report_missing_val = subgroup_report(df_val_analysis, 'missing_high')
report_missing_test = subgroup_report(df_test_analysis, 'missing_high')

report_hr_val = subgroup_report(df_val_analysis, 'hr_high')
report_hr_test = subgroup_report(df_test_analysis, 'hr_high')

print("Validation - EDA pattern strong")
display(report_pattern_val)

print("Test - EDA pattern strong")
display(report_pattern_test)

print("Validation - Missing high")
display(report_missing_val)

print("Test - Missing high")
display(report_missing_test)

print("Validation - HR high")
display(report_hr_val)

print("Test - HR high")
display(report_hr_test)

Validation - EDA pattern strong


,group,n,positive_rate,AUPRC,sensitivity,specificity,precision,f1,FP,FN,TP,TN,mean_prob,mean_member_std
0,0,32041,0.073624,0.383811,0.799915,0.777104,0.221922,0.347450,6616,472,1887,23066,0.206465,0.085820
1,1,4556,0.095259,0.444443,0.808756,0.771713,0.271672,0.406721,941,83,351,3181,0.220620,0.087718


Test - EDA pattern strong


,group,n,positive_rate,AUPRC,sensitivity,specificity,precision,f1,FP,FN,TP,TN,mean_prob,mean_member_std
0,0,39956,0.068275,0.417209,0.850440,0.782637,0.222820,0.353120,8092,408,2320,29136,0.204306,0.084758
1,1,5596,0.087026,0.429555,0.815195,0.774320,0.256129,0.389789,1153,90,397,3956,0.219417,0.086038


Validation - Missing high


...

,group,n,positive_rate,AUPRC,sensitivity,specificity,precision,f1,FP,FN,TP,TN,mean_prob,mean_member_std
0,0,21859,0.078183,0.420386,0.861908,0.750571,0.226650,0.358918,5026,236,1473,15124,0.232193,0.089879
1,1,23693,0.063563,0.418105,0.826029,0.809844,0.227714,0.357010,4219,262,1244,17968,0.182146,0.080335


Validation - HR high


,group,n,positive_rate,AUPRC,sensitivity,specificity,precision,f1,FP,FN,TP,TN,mean_prob,mean_member_std
0,0,30182,0.066497,0.359452,0.798206,0.797090,0.218882,0.343556,5717,405,1602,22458,0.190614,0.081290
1,1,6415,0.122525,0.473300,0.809160,0.673121,0.256866,0.389945,1840,150,636,3789,0.291093,0.108484


Test - HR high


,group,n,positive_rate,AUPRC,sensitivity,specificity,precision,f1,FP,FN,TP,TN,mean_prob,mean_member_std
0,0,37614,0.061759,0.387596,0.822643,0.804483,0.216888,0.343273,6900,412,1911,28391,0.186300,0.079962
1,1,7938,0.112371,0.493740,0.903587,0.667187,0.255792,0.398714,2345,86,806,4701,0.300279,0.108384


In [22]:
print("Current validation error counts:")
print(df_val_analysis['error_type'].value_counts())

print("\nCurrent test error counts:")
print(df_test_analysis['error_type'].value_counts())

Current validation error counts:
error_type
TN    26247
FP     7557
TP     2238
FN      555
Name: count, dtype: int64

Current test error counts:
error_type
TN    33092
FP     9245
TP     2717
FN      498
Name: count, dtype: int64


In [23]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    df_test_analysis['y_true'],
    df_test_analysis['y_pred'],
    labels=[0, 1]
).ravel()

print("TN, FP, FN, TP =", tn, fp, fn, tp)

TN, FP, FN, TP = 33092 9245 498 2717


### FN và FP theo subgroup.

In [24]:
def fn_report(df, group_col):
    rows = []

    for group_value, sub in df.groupby(group_col):
        pos = sub[sub['y_true'] == 1].copy()   # chỉ xét các cửa sổ thật sự positive
        n_pos = len(pos)

        fn = (pos['error_type'] == 'FN').sum()
        tp = (pos['error_type'] == 'TP').sum()

        fn_rate = fn / n_pos if n_pos > 0 else np.nan
        tp_rate = tp / n_pos if n_pos > 0 else np.nan

        fn_sub = pos[pos['error_type'] == 'FN']

        rows.append({
            'group': group_value,
            'n_total': len(sub),
            'n_positive': n_pos,
            'FN': fn,
            'TP': tp,
            'FN_rate_within_positive': fn_rate,
            'TP_rate_within_positive': tp_rate,
            'mean_prob_positive': pos['y_prob'].mean() if n_pos > 0 else np.nan,
            'mean_prob_FN': fn_sub['y_prob'].mean() if len(fn_sub) > 0 else np.nan,
            'mean_member_std_FN': fn_sub['member_prob_std'].mean() if len(fn_sub) > 0 else np.nan,
        })

    return pd.DataFrame(rows).sort_values('group')

In [25]:
def fp_report(df, group_col):
    rows = []

    for group_value, sub in df.groupby(group_col):
        neg = sub[sub['y_true'] == 0].copy()   # chỉ xét các cửa sổ thật sự negative
        n_neg = len(neg)

        fp = (neg['error_type'] == 'FP').sum()
        tn = (neg['error_type'] == 'TN').sum()

        fp_rate = fp / n_neg if n_neg > 0 else np.nan
        tn_rate = tn / n_neg if n_neg > 0 else np.nan

        fp_sub = neg[neg['error_type'] == 'FP']

        rows.append({
            'group': group_value,
            'n_total': len(sub),
            'n_negative': n_neg,
            'FP': fp,
            'TN': tn,
            'FP_rate_within_negative': fp_rate,
            'TN_rate_within_negative': tn_rate,
            'mean_prob_negative': neg['y_prob'].mean() if n_neg > 0 else np.nan,
            'mean_prob_FP': fp_sub['y_prob'].mean() if len(fp_sub) > 0 else np.nan,
            'mean_member_std_FP': fp_sub['member_prob_std'].mean() if len(fp_sub) > 0 else np.nan,
        })

    return pd.DataFrame(rows).sort_values('group')

In [26]:
# ===== FN reports on test =====
fn_pattern_test = fn_report(df_test_analysis, 'eda_pattern_strong')
fn_missing_test = fn_report(df_test_analysis, 'missing_high')
fn_hr_test = fn_report(df_test_analysis, 'hr_high')
fn_resp_test = fn_report(df_test_analysis, 'resp_high')
fn_sbp_test = fn_report(df_test_analysis, 'sbp_low')
fn_map_test = fn_report(df_test_analysis, 'map_low')

print("Test - FN by EDA pattern strong")
display(fn_pattern_test)

print("Test - FN by Missing high")
display(fn_missing_test)

print("Test - FN by HR high")
display(fn_hr_test)

print("Test - FN by Resp high")
display(fn_resp_test)

print("Test - FN by SBP low")
display(fn_sbp_test)

print("Test - FN by MAP low")
display(fn_map_test)

Test - FN by EDA pattern strong


,group,n_total,n_positive,FN,TP,FN_rate_within_positive,TP_rate_within_positive,mean_prob_positive,mean_prob_FN,mean_member_std_FN
0,0,39956,2728,408,2320,0.149560,0.850440,0.663616,0.124863,0.120863
1,1,5596,487,90,397,0.184805,0.815195,0.642926,0.117961,0.113579


Test - FN by Missing high


,group,n_total,n_positive,FN,TP,FN_rate_within_positive,TP_rate_within_positive,mean_prob_positive,mean_prob_FN,mean_member_std_FN
0,0,21859,1709,236,1473,0.138092,0.861908,0.678445,0.135926,0.130832
1,1,23693,1506,262,1244,0.173971,0.826029,0.640098,0.112528,0.109381


Test - FN by HR high


...

,group,n_total,n_positive,FN,TP,FN_rate_within_positive,TP_rate_within_positive,mean_prob_positive,mean_prob_FN,mean_member_std_FN
0,0,35296,2088,390,1698,0.186782,0.813218,0.628810,0.120929,0.117273
1,1,10256,1127,108,1019,0.095830,0.904170,0.719161,0.133321,0.127757


Test - FN by SBP low


,group,n_total,n_positive,FN,TP,FN_rate_within_positive,TP_rate_within_positive,mean_prob_positive,mean_prob_FN,mean_member_std_FN
0,0,39435,2661,438,2223,0.164600,0.835400,0.659560,0.124412,0.119647
1,1,6117,554,60,494,0.108303,0.891697,0.664909,0.117804,0.118812


Test - FN by MAP low


,group,n_total,n_positive,FN,TP,FN_rate_within_positive,TP_rate_within_positive,mean_prob_positive,mean_prob_FN,mean_member_std_FN
0,0,40912,2759,452,2307,0.163827,0.836173,0.656626,0.123927,0.119219
1,1,4640,456,46,410,0.100877,0.899123,0.683811,0.120563,0.122764


In [27]:
# ===== FP reports on test =====
fp_pattern_test = fp_report(df_test_analysis, 'eda_pattern_strong')
fp_missing_test = fp_report(df_test_analysis, 'missing_high')
fp_hr_test = fp_report(df_test_analysis, 'hr_high')
fp_resp_test = fp_report(df_test_analysis, 'resp_high')
fp_sbp_test = fp_report(df_test_analysis, 'sbp_low')
fp_map_test = fp_report(df_test_analysis, 'map_low')

print("Test - FP by EDA pattern strong")
display(fp_pattern_test)

print("Test - FP by Missing high")
display(fp_missing_test)

print("Test - FP by HR high")
display(fp_hr_test)

print("Test - FP by Resp high")
display(fp_resp_test)

print("Test - FP by SBP low")
display(fp_sbp_test)

print("Test - FP by MAP low")
display(fp_map_test)

Test - FP by EDA pattern strong


,group,n_total,n_negative,FP,TN,FP_rate_within_negative,TN_rate_within_negative,mean_prob_negative,mean_prob_FP,mean_member_std_FP
0,0,39956,37228,8092,29136,0.217363,0.782637,0.170648,0.589341,0.182486
1,1,5596,5109,1153,3956,0.225680,0.774320,0.179047,0.602182,0.178496


Test - FP by Missing high


,group,n_total,n_negative,FP,TN,FP_rate_within_negative,TN_rate_within_negative,mean_prob_negative,mean_prob_FP,mean_member_std_FP
0,0,21859,20150,5026,15124,0.249429,0.750571,0.194345,0.606618,0.176210
1,1,23693,22187,4219,17968,0.190156,0.809844,0.151061,0.572269,0.188873


Test - FP by HR high


...

,group,n_total,n_negative,FP,TN,FP_rate_within_negative,TN_rate_within_negative,mean_prob_negative,mean_prob_FP,mean_member_std_FP
0,0,35296,33208,6407,26801,0.192935,0.807065,0.153729,0.578739,0.188206
1,1,10256,9129,2838,6291,0.310877,0.689123,0.236896,0.618494,0.167953


Test - FP by SBP low


,group,n_total,n_negative,FP,TN,FP_rate_within_negative,TN_rate_within_negative,mean_prob_negative,mean_prob_FP,mean_member_std_FP
0,0,39435,36774,7670,29104,0.208571,0.791429,0.165457,0.592465,0.179659
1,1,6117,5563,1575,3988,0.283121,0.716879,0.212681,0.583531,0.193334


Test - FP by MAP low


,group,n_total,n_negative,FP,TN,FP_rate_within_negative,TN_rate_within_negative,mean_prob_negative,mean_prob_FP,mean_member_std_FP
0,0,40912,38153,8010,30143,0.209944,0.790056,0.166322,0.593919,0.181119
1,1,4640,4184,1235,2949,0.295172,0.704828,0.220356,0.571639,0.187626
